# 02 — Prepare canonical analysis inputs

Load Notebook 01's authoritative corrected Stream, calibrated StationXML, and
master channel summary. Derive channel membership, units, geometry, array
reference, and the infrasound reference channel from those products rather
than maintaining a second hard-coded channel inventory.

## 1. Imports and authoritative inputs

In [1]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import median_filter

from obspy import read, read_inventory
from pyproj import Transformer

project_root = Path.cwd().resolve()
if project_root.name == "notebooks2":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules import project_config as config
from modules.geometry_products import write_geometry_products
from modules.kml_utils import read_kml_points
from modules.xml_utils import inventory_stations_to_dataframe

READ_START = config.EXPLOSION_TIME - config.PRETRIGGER_WINDOW
READ_END = config.EXPLOSION_TIME + config.POSTTRIGGER_WINDOW

required_products = {
    "corrected Pickle": config.FINAL_CORRECTED_PICKLE,
    "calibrated StationXML": config.FINAL_INVENTORY_XML,
    "master channel summary": config.FINAL_CORRECTION_SUMMARY,
    "launchpad/camera KML": config.KML_FILE,
}
for label, path in required_products.items():
    if not Path(path).exists():
        raise FileNotFoundError(
            f"{label} not found: {path}. Run Notebook 01 first."
        )

st_corr = read(
    str(config.FINAL_CORRECTED_PICKLE),
    format="PICKLE",
)
st_corr.trim(READ_START, READ_END, pad=False)
st_corr.sort(keys=["channel"])

inventory_event = read_inventory(
    str(config.FINAL_INVENTORY_XML)
)
channel_summary = pd.read_csv(
    config.FINAL_CORRECTION_SUMMARY
)

## 2. Discover and validate channels

The corrected Stream supplies the actual NSLC identifiers and physical units.
The master CSV supplies the corresponding geometry and processing metadata.
The two products must contain exactly the same unique SEED identifiers.

In [2]:
trace_rows = []
for trace in st_corr:
    units = trace.stats.get("units")
    if not units:
        raise ValueError(f"Missing physical units on {trace.id}")
    trace_rows.append({
        "seed_id": trace.id,
        "network": trace.stats.network,
        "station": trace.stats.station,
        "location": trace.stats.location,
        "channel": trace.stats.channel,
        "sampling_rate_hz": float(trace.stats.sampling_rate),
        "units": str(units),
    })

trace_metadata = pd.DataFrame(trace_rows)
for table_name, table in (
    ("corrected Stream", trace_metadata),
    ("master channel summary", channel_summary),
):
    if table["seed_id"].duplicated().any():
        duplicates = table.loc[
            table["seed_id"].duplicated(keep=False),
            "seed_id",
        ].tolist()
        raise ValueError(
            f"Duplicate identifiers in {table_name}: {duplicates}"
        )

stream_ids = set(trace_metadata["seed_id"])
summary_ids = set(channel_summary["seed_id"])
if stream_ids != summary_ids:
    raise ValueError(
        "Notebook 01 products disagree: "
        f"missing from summary={sorted(stream_ids - summary_ids)}, "
        f"missing from Stream={sorted(summary_ids - stream_ids)}"
    )

def normalized_units(value: str) -> str:
    return value.strip().lower().replace(" ", "")


unit_kind = trace_metadata["units"].map(normalized_units)
pressure_mask = unit_kind.eq("pa")
seismic_mask = unit_kind.isin({"m/s", "m/s**1", "m*s^-1"})
if (~(pressure_mask | seismic_mask)).any():
    unsupported = trace_metadata.loc[
        ~(pressure_mask | seismic_mask),
        ["seed_id", "units"],
    ]
    raise ValueError(
        "Unsupported corrected units:\n"
        + unsupported.to_string(index=False)
    )

channels = trace_metadata["channel"].tolist()
pressure_channels = trace_metadata.loc[
    pressure_mask, "channel"
].tolist()
seismic_channels = trace_metadata.loc[
    seismic_mask, "channel"
].tolist()
if not pressure_channels or not seismic_channels:
    raise ValueError(
        "Expected both pressure and seismic channels; "
        f"pressure={pressure_channels}, seismic={seismic_channels}"
    )

def unique_value(dataframe: pd.DataFrame, column: str):
    values = dataframe[column].dropna().unique()
    if len(values) != 1:
        raise ValueError(
            f"Expected one {column}; found {values.tolist()}"
        )
    return values[0]


network = str(unique_value(trace_metadata, "network"))
station = str(unique_value(trace_metadata, "station"))
location = str(unique_value(trace_metadata, "location"))
sampling_rate_hz = float(
    unique_value(trace_metadata, "sampling_rate_hz")
)

print("Channels:", channels)
print("Pressure channels:", pressure_channels)
print("Seismic channels:", seismic_channels)
print("NSL:", network, station, location)

Channels: ['DD1', 'DD2', 'DD3', 'DHE', 'DHN', 'DHZ']
Pressure channels: ['DD1', 'DD2', 'DD3']
Seismic channels: ['DHE', 'DHN', 'DHZ']
NSL: 1R BCHH 10


## 3. Save key-event definitions

In [3]:
key_events_file = config.OUTPUT_DIR / "02_key_events.csv"
config.events.to_csv(key_events_file, index=False)
display(config.events)
print(f"Wrote {key_events_file}")

,event_id,label,source_time
0,upper_stage,Upper Stage,2016-09-01T13:07:12.080000Z
1,lower_stage,Lower Stage,2016-09-01T13:07:15.750000Z
2,capsule_impact,Capsule Impact,2016-09-01T13:07:24.600000Z
3,capsule_explosion,Capsule Explosion,2016-09-01T13:07:25.150000Z


Wrote /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/02_key_events.csv


## 4. Derive receiver and source geometry

Geographic coordinates, source distances, and azimuths come from Notebook 01.
Notebook 02 adds UTM Zone 17N coordinates. Sensor type is inferred from
physical units, and channel order follows the corrected Stream.

In [4]:
required_geometry_columns = {
    "seed_id",
    "network",
    "station",
    "location",
    "channel",
    "latitude",
    "longitude",
    "elevation_m",
    "depth_m",
    "azimuth_deg",
    "dip_deg",
    "sample_rate_hz",
    "source_name",
    "source_latitude",
    "source_longitude",
    "source_elevation_m",
    "source_to_sensor_distance_m",
    "source_to_sensor_azimuth_deg",
    "sensor_to_source_back_azimuth_deg",
}
missing = required_geometry_columns - set(channel_summary.columns)
if missing:
    raise KeyError(
        "Notebook 01 summary lacks geometry columns: "
        f"{sorted(missing)}"
    )

# Preserve the Stream order without encoding channel names in this notebook.
stream_order = {
    seed_id: index
    for index, seed_id in enumerate(trace_metadata["seed_id"])
}
bchh_geometry = channel_summary.copy()
bchh_geometry["_stream_order"] = (
    bchh_geometry["seed_id"].map(stream_order)
)
bchh_geometry = (
    bchh_geometry
    .sort_values("_stream_order")
    .drop(columns="_stream_order")
    .reset_index(drop=True)
)

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:32617",
    always_xy=True,
)
easting_m, northing_m = transformer.transform(
    bchh_geometry["longitude"].to_numpy(float),
    bchh_geometry["latitude"].to_numpy(float),
)
bchh_geometry["easting_m"] = easting_m
bchh_geometry["northing_m"] = northing_m
bchh_geometry["distance_m"] = (
    bchh_geometry["source_to_sensor_distance_m"]
)
bchh_geometry["source_to_receiver_azimuth_deg"] = (
    bchh_geometry["source_to_sensor_azimuth_deg"]
)
bchh_geometry["back_azimuth_to_slc40_deg"] = (
    bchh_geometry["sensor_to_source_back_azimuth_deg"]
)

units_by_seed_id = dict(
    zip(trace_metadata["seed_id"], trace_metadata["units"])
)
bchh_geometry["physical_units"] = (
    bchh_geometry["seed_id"].map(units_by_seed_id)
)
bchh_geometry["sensor"] = np.where(
    bchh_geometry["physical_units"].map(normalized_units).eq("pa"),
    "infraBSU " + bchh_geometry["channel"],
    "Trillium Compact",
)
bchh_geometry["label"] = np.where(
    bchh_geometry["physical_units"].map(normalized_units).eq("pa"),
    bchh_geometry["channel"],
    "BCHH",
)

# Collapse the three colocated seismic components to one physical sensor.
bchh_sensors_df = (
    bchh_geometry
    .rename(columns={"latitude": "lat", "longitude": "lon"})
    .drop_duplicates(subset="sensor")
    [
        [
            "sensor",
            "label",
            "easting_m",
            "northing_m",
            "lat",
            "lon",
        ]
    ]
    .rename(
        columns={
            "easting_m": "easting",
            "northing_m": "northing",
        }
    )
    .reset_index(drop=True)
)

display(bchh_geometry)
display(bchh_sensors_df)

,seed_id,network,station,location,channel,latitude,longitude,elevation_m,depth_m,azimuth_deg,...,corrected_median,corrected_standard_deviation,corrected_rms,easting_m,northing_m,distance_m,source_to_receiver_azimuth_deg,back_azimuth_to_slc40_deg,sensor,label
0,1R.BCHH.10.DD1,1R,BCHH,10,DD1,28.574222,-80.572417,0.0,0.0,0.0,...,-4.131989e-01,33.066901,33.066901,541816.300901,3.160889e+06,1439.963831,18.985737,198.988027,infraBSU DD1,DD1
1,1R.BCHH.10.DD2,1R,BCHH,10,DD2,28.573881,-80.572296,0.0,0.0,90.0,...,-4.250432e-02,7.850680,7.850681,541828.276924,3.160851e+06,1408.338457,19.940904,199.943252,infraBSU DD2,DD2
2,1R.BCHH.10.DD3,1R,BCHH,10,DD3,28.574006,-80.572560,0.0,0.0,0.0,...,1.222732e+00,13.326245,13.326245,541802.343163,3.160865e+06,1412.854330,18.761692,198.763913,infraBSU DD3,DD3
3,1R.BCHH.10.DHE,1R,BCHH,10,DHE,28.574017,-80.572376,0.0,0.0,90.0,...,-1.189176e-09,0.000013,0.000013,541820.377763,3.160867e+06,1419.886511,19.435717,199.438026,Trillium Compact,BCHH
4,1R.BCHH.10.DHN,1R,BCHH,10,DHN,28.574017,-80.572376,0.0,0.0,0.0,...,1.329465e-11,0.000022,0.000022,541820.377763,3.160867e+06,1419.886511,19.435717,199.438026,Trillium Compact,BCHH
5,1R.BCHH.10.DHZ,1R,BCHH,10,DHZ,28.574017,-80.572376,0.0,0.0,0.0,...,-2.071039e-09,0.000033,0.000033,541820.377763,3.160867e+06,1419.886511,19.435717,199.438026,Trillium Compact,BCHH


,sensor,label,easting,northing,lat,lon
0,infraBSU DD1,DD1,541816.300901,3.160889e+06,28.574222,-80.572417
1,infraBSU DD2,DD2,541828.276924,3.160851e+06,28.573881,-80.572296
2,infraBSU DD3,DD3,541802.343163,3.160865e+06,28.574006,-80.572560
3,Trillium Compact,BCHH,541820.377763,3.160867e+06,28.574017,-80.572376


## 5. Derive array and infrasound reference locations

The array reference is the common seismic-sensor location. The infrasound
reference is selected as the pressure sensor nearest that location, replacing
the previously hard-coded reference-channel choice.

In [5]:
seismic_geometry = bchh_geometry.loc[
    bchh_geometry["channel"].isin(seismic_channels)
].copy()
pressure_geometry = bchh_geometry.loc[
    bchh_geometry["channel"].isin(pressure_channels)
].copy()

seismic_locations = seismic_geometry[
    ["latitude", "longitude", "easting_m", "northing_m"]
].drop_duplicates()
if len(seismic_locations) != 1:
    raise ValueError(
        "Expected the seismic components to share one location; "
        f"found {len(seismic_locations)}"
    )

array_location = seismic_locations.iloc[0]
array_easting_m = float(array_location["easting_m"])
array_northing_m = float(array_location["northing_m"])

pressure_geometry["distance_to_array_center_m"] = np.hypot(
    pressure_geometry["easting_m"] - array_easting_m,
    pressure_geometry["northing_m"] - array_northing_m,
)
infrasound_reference = pressure_geometry.loc[
    pressure_geometry["distance_to_array_center_m"].idxmin()
]
infrasound_reference_channel = str(
    infrasound_reference["channel"]
)

# All colocated seismic rows should carry the same source geometry.
for column in (
    "distance_m",
    "back_azimuth_to_slc40_deg",
):
    if seismic_geometry[column].nunique(dropna=False) != 1:
        raise ValueError(
            f"Seismic components disagree on {column}"
        )
array_reference = seismic_geometry.iloc[0]

print(
    "Infrasound reference:",
    infrasound_reference_channel,
    f"({infrasound_reference['distance_to_array_center_m']:.2f} m "
    "from the seismometer)",
)

Infrasound reference: DD2 (16.99 m from the seismometer)


## 6. Apply the adopted infrasound moving-median baseline correction

The adopted pressure preprocessing is applied here, before catalogue validation
and review. Only the pressure channels are changed; the three seismic channels
are copied unchanged.

The baseline is a centered 1.0 s moving median. Notebook 06 later validates this
fixed choice against alternative windows and quantifies its effect on event
measurements. Notebook 06 does not create the canonical corrected Stream.


In [6]:
MOVING_MEDIAN_WINDOW_S = 1.0


def odd_sample_count(duration_s: float, sampling_rate_hz: float) -> int:
    count = max(3, int(round(duration_s * sampling_rate_hz)))
    if count % 2 == 0:
        count += 1
    return count


def moving_median_baseline(
    values: np.ndarray,
    *,
    sampling_rate_hz: float,
    window_s: float,
) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    return median_filter(
        values,
        size=odd_sample_count(window_s, sampling_rate_hz),
        mode="nearest",
    )


st_baseline_corrected = st_corr.__class__()
for trace in st_corr:
    corrected = trace.copy()
    if trace.stats.channel in pressure_channels:
        baseline = moving_median_baseline(
            trace.data,
            sampling_rate_hz=float(trace.stats.sampling_rate),
            window_s=MOVING_MEDIAN_WINDOW_S,
        )
        corrected.data = trace.data.astype(float) - baseline
        corrected.stats.processing = list(
            getattr(corrected.stats, "processing", [])
        )
        corrected.stats.processing.append(
            "centered moving-median baseline removed: "
            f"window={MOVING_MEDIAN_WINDOW_S:g} s"
        )
    st_baseline_corrected += corrected

st_baseline_corrected.sort(keys=["channel"])

if [tr.stats.channel for tr in st_baseline_corrected] != sorted(channels):
    raise ValueError("Baseline-corrected Stream channel set/order is unexpected")

print(st_baseline_corrected)


6 Trace(s) in Stream:
1R.BCHH.10.DD1 | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
1R.BCHH.10.DD2 | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
1R.BCHH.10.DD3 | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
1R.BCHH.10.DHE | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
1R.BCHH.10.DHN | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
1R.BCHH.10.DHZ | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples


## 7. Write the canonical Notebook 02 products


In [7]:
kml_points = read_kml_points(config.KML_FILE)
locations_df = (
    pd.DataFrame.from_dict(kml_points, orient="index")
    .rename_axis("kml_id")
    .reset_index()
)
stations_df = inventory_stations_to_dataframe(inventory_event)

geometry_paths = write_geometry_products(
    inventory=inventory_event,
    channels_df=bchh_geometry,
    stations_df=stations_df,
    locations_df=locations_df,
    output_directory=config.OUTPUT_DIR,
    prefix="02_",
)
sensor_file = config.OUTPUT_DIR / "02_bchh_physical_sensors.csv"
bchh_sensors_df.to_csv(sensor_file, index=False)

stream_file = (
    config.OUTPUT_DIR
    / "02_bchh_corrected_analysis_window.pkl"
)
st_corr.write(str(stream_file), format="PICKLE")

config.DERIVED_DIR.mkdir(parents=True, exist_ok=True)
baseline_pickle_file = (
    config.DERIVED_DIR
    / "bchh_corrected_moving_median_baseline_removed.pkl"
)
baseline_mseed_file = (
    config.DERIVED_DIR
    / "bchh_corrected_moving_median_baseline_removed.mseed"
)

# PICKLE preserves ObsPy metadata within the project Python environment.
with baseline_pickle_file.open("wb") as file_object:
    pickle.dump(
        st_baseline_corrected,
        file_object,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

# MiniSEED is the portable waveform interchange product used by S03/InfraPy.
st_baseline_corrected.write(
    str(baseline_mseed_file),
    format="MSEED",
)

configuration_file = (
    config.OUTPUT_DIR
    / "02_analysis_configuration.json"
)
analysis_configuration = {
    "schema_version": 4,
    "network": network,
    "station": station,
    "location": location,
    "channels": channels,
    "pressure_channels": pressure_channels,
    "seismic_channels": seismic_channels,
    "sampling_rate_hz": sampling_rate_hz,
    "analysis_start_epoch_s": float(READ_START.timestamp),
    "analysis_end_epoch_s": float(READ_END.timestamp),
    "slc40": {
        "name": str(
            unique_value(bchh_geometry, "source_name")
        ),
        "latitude": float(
            unique_value(bchh_geometry, "source_latitude")
        ),
        "longitude": float(
            unique_value(bchh_geometry, "source_longitude")
        ),
        "elevation_m": float(
            unique_value(bchh_geometry, "source_elevation_m")
        ),
    },
    "array_reference": {
        "name": "BCHH seismometer",
        "latitude": float(array_location["latitude"]),
        "longitude": float(array_location["longitude"]),
        "easting_m": array_easting_m,
        "northing_m": array_northing_m,
        "distance_m": float(array_reference["distance_m"]),
        "back_azimuth_to_slc40_deg": float(
            array_reference["back_azimuth_to_slc40_deg"]
        ),
    },
    "infrasound_reference_channel": (
        infrasound_reference_channel
    ),
    "notebook_01_products": {
        "channel_summary": str(
            config.FINAL_CORRECTION_SUMMARY
        ),
        "corrected_pickle": str(
            config.FINAL_CORRECTED_PICKLE
        ),
        "calibrated_stationxml": str(
            config.FINAL_INVENTORY_XML
        ),
    },
    "geometry_file": str(geometry_paths["channels"]),
    "physical_sensors_file": str(sensor_file),
    "stream_file": str(stream_file),
    "baseline_removed_streams": {
        "pickle": str(baseline_pickle_file),
        "mseed": str(baseline_mseed_file),
    },
    "moving_median_baseline": {
        "method": "centered moving median",
        "window_s": MOVING_MEDIAN_WINDOW_S,
        "applied_to_channels": list(pressure_channels),
        "unchanged_channels": list(seismic_channels),
        "producer_notebook": "02_prepare_analysis_inputs.ipynb",
    },
    "key_events_file": str(key_events_file),
}
configuration_file.write_text(
    json.dumps(analysis_configuration, indent=2) + "\n",
    encoding="utf-8",
)

outputs = {
    **geometry_paths,
    "physical_sensors": sensor_file,
    "analysis_stream": stream_file,
    "baseline_removed_pickle": baseline_pickle_file,
    "baseline_removed_mseed": baseline_mseed_file,
    "key_events": key_events_file,
    "analysis_configuration": configuration_file,
}
for name, path in outputs.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
    print(f"{name}: {path}")

inventory: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/02_BCHH_20160901_event_inventory.xml
channels: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/02_bchh_channels.csv
stations: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/02_bchh_stations.csv
locations: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/02_launchpad_camera_locations.csv
physical_sensors: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/02_bchh_physical_sensors.csv
analysis_stream: /Users/thompsong/Library/CloudStorag

/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1034: UserWarning: The encoding specified in trace.stats.mseed.encoding does not match the dtype of the data.
A suitable encoding will be chosen.
  warnings.warn(msg, UserWarning)
